<a href="https://colab.research.google.com/github/samkenzie/predicting-electrode-transport-properties/blob/main/Baseline2_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Model 2: Random Forest

The purpose of the random forest model is to see if a machine learning model can interpret more complex relations from not just porosity, as is used by the Bruggeman relation, but also with particle count and particle boundary length.

The procedure for this baseline model is as follows:
1. Import image data
2. Obtain porosity from image sample (ratio of white vs. total pixels in the 64x64 image)
3. Import effective diffusivity and conductivity, order to match image data

*Thus far 1-3 are an exact copy of Baseline Model 1: Bruggeman... :)
4.

In [3]:
#0. Imports
import numpy as np
from pathlib import Path
import glob
from PIL import Image
from scipy import ndimage #https://docs.scipy.org/doc/scipy/reference/ndimage.html
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [4]:
#1. Import image data
!unzip -q /content/images_aps360.zip -d /content/images
!unzip -q /content/diffusion_aps360.zip -d /content/texts

In [5]:
img_ids, img_pixels = [], []
for path in glob.glob("/content/images/64/*.tif"):
    img_ids.append(int(Path(path).stem)) #123.tif -> 123
    img_pixels.append(np.array(Image.open(path).convert("L"), dtype=np.uint8))

img_ids = np.array(img_ids)
img_pixels = np.stack(img_pixels) #Expected (100000, 64, 64)

print(f"Loaded {len(img_ids)} images")
print(f"Array shape {img_pixels.shape}")

Loaded 100000 images
Array shape (100000, 64, 64)


In [6]:
#2. Obtain porosity
porosity = []

for img in img_pixels: #ie. for one 64x64 image
    num_white = (img == 255).sum() #Sum of pixels that =255 (aka. white)
    porosity.append(num_white / img.size) #img.size = 64*64 = 4096

porosity = np.array(porosity)

print(f"Calculated {len(porosity)} porosities")
print(f"Range: {porosity.min():.3f} to {porosity.max():.3f}, mean {porosity.mean():.3f}, median {np.median(porosity):.3f}")

Calculated 100000 porosities
Range: 0.153 to 0.898, mean 0.492, median 0.491


In [7]:
#3. Import effective diffusivity and conductivity data
txt_ids, eff_diff, eff_cond = [], [], []

for path in glob.glob("/content/texts/dispersion/*.txt"):
    with open(path) as fh:
        vals = fh.read().split()
        txt_ids.append(int(Path(path).stem))
        eff_diff.append(float(vals[0]))
        eff_cond.append(float(vals[1]))

print(f"Loaded {len(txt_ids)} text files")

Loaded 100000 text files


In [8]:
#3. CONT. Ordering targets to match image data order
order = {id_: i for i, id_ in enumerate(txt_ids)}
id = [order[i] for i in img_ids]
eff_diff = np.array(eff_diff)[id]
eff_cond = np.array(eff_cond)[id]

In [9]:
txt_ids_aligned = np.array(txt_ids)[id] #Grabbing .txt IDs

print("img_id  txt_id")
for i in range(10): #Checking if the first 10 image and text file IDs match
    print(f"{img_ids[i]:6d} {txt_ids_aligned[i]:6d}")

print(np.all(img_ids == txt_ids_aligned)) #True if all IDs match

img_id  txt_id
 83079  83079
 54861  54861
 35872  35872
  1383   1383
 68043  68043
 46226  46226
 83532  83532
 69032  69032
 17618  17618
 15812  15812
True


Now, onto the new stuff for this baseline model!

In [10]:
#4. Determine number of particles per image
num_particles = []

for img in img_pixels: #ie. for one 64x64 image
    labeled, num = ndimage.label(img == 0) #Labelling interconnected regions of black pixels
    num_particles.append(num)

num_particles = np.array(num_particles)

print(f"Calculated {len(num_particles)} particle counts")
print("Range:", num_particles.min(), "to", num_particles.max()) #Expected 1-20 as indicated by dataset

Calculated 100000 particle counts
Range: 1 to 20


In [11]:
#5. Determine total particle boundary length per image
edges = []

for img in img_pixels: #ie. for one 64x64 image
    horiz = (img[:, 1:] != img[:, :-1]).sum() #Pixel vs. right neighbour
    vert = (img[1:, :] != img[:-1, :]).sum() #Pixel vs. neighbour below
    edges.append(horiz + vert)

edges = np.array(edges)

print(f"Calculated {len(edges)} edge counts")
print("Range:", edges.min(), "to", edges.max())

Calculated 100000 edge counts
Range: 122 to 934


In [12]:
#6. Training basic random forest
X = np.column_stack([porosity, num_particles, edges]) #Features
y = np.column_stack([eff_diff, eff_cond]) #Targets

In [13]:
#Train test split (70/15/15)
size_test = int(0.15 * len(porosity))
size_val = int(0.15 * len(porosity))
train_val, test = train_test_split(np.arange(len(porosity)), test_size=size_test, random_state=0)
train, val = train_test_split(train_val, test_size=size_val, random_state=0)
X_train, X_val, X_test = X[train], X[val], X[test]
y_train, y_val, y_test = y[train], y[val], y[test]
print(len(train), len(val), len(test)) # Expected 70000 15000 15000

70000 15000 15000


In [14]:
for leaf in [1, 5, 20, 50, 100, 200]:
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf, n_jobs=-1, random_state=0)
    rf.fit(X_train, y_train)
    p = rf.predict(X_val)
    print(f"min_samples_leaf={leaf:4d}   "
          f"RF R^2 effective diffusivity: {r2_score(y_val[:, 0], p[:, 0]):.3f}  "
          f"RF R^2 effective conductivity: {r2_score(y_val[:, 1], p[:, 1]):.3f}")

min_samples_leaf=   1   RF R^2 effective diffusivity: 0.843  RF R^2 effective conductivity: 0.839
min_samples_leaf=   5   RF R^2 effective diffusivity: 0.862  RF R^2 effective conductivity: 0.859
min_samples_leaf=  20   RF R^2 effective diffusivity: 0.866  RF R^2 effective conductivity: 0.864
min_samples_leaf=  50   RF R^2 effective diffusivity: 0.867  RF R^2 effective conductivity: 0.864
min_samples_leaf= 100   RF R^2 effective diffusivity: 0.866  RF R^2 effective conductivity: 0.863
min_samples_leaf= 200   RF R^2 effective diffusivity: 0.866  RF R^2 effective conductivity: 0.863


In [15]:
for depth in [None, 4, 8, 12, 16]:
    rf = RandomForestRegressor(n_estimators=100, max_depth=depth, n_jobs=-1, random_state=0)
    rf.fit(X_train, y_train)
    p = rf.predict(X_val)
    depth_str = "None" if depth is None else str(depth)
    print(f"max_depth={depth_str:>4s}  "
          f"RF R^2 effective diffusivity: {r2_score(y_val[:, 0], p[:, 0]):.3f}  "
          f"RF R^2 effective conductivity: {r2_score(y_val[:, 1], p[:, 1]):.3f}")

max_depth=None  RF R^2 effective diffusivity: 0.843  RF R^2 effective conductivity: 0.839
max_depth=   4  RF R^2 effective diffusivity: 0.862  RF R^2 effective conductivity: 0.858
max_depth=   8  RF R^2 effective diffusivity: 0.866  RF R^2 effective conductivity: 0.864
max_depth=  12  RF R^2 effective diffusivity: 0.866  RF R^2 effective conductivity: 0.864
max_depth=  16  RF R^2 effective diffusivity: 0.861  RF R^2 effective conductivity: 0.859


In [16]:
for num in [10, 50, 100, 200, 500]:
    rf = RandomForestRegressor(n_estimators=num, n_jobs=-1, random_state=0)
    rf.fit(X_train, y_train)
    p = rf.predict(X_val)
    print(f"n_estimators={num:6d}  "
          f"RF R^2 effective diffusivity: {r2_score(y_val[:, 0], p[:, 0]):.3f}  "
          f"RF R^2 effective conductivity: {r2_score(y_val[:, 1], p[:, 1]):.3f}")

n_estimators=    10  RF R^2 effective diffusivity: 0.833  RF R^2 effective conductivity: 0.828
n_estimators=    50  RF R^2 effective diffusivity: 0.842  RF R^2 effective conductivity: 0.838
n_estimators=   100  RF R^2 effective diffusivity: 0.843  RF R^2 effective conductivity: 0.839
n_estimators=   200  RF R^2 effective diffusivity: 0.843  RF R^2 effective conductivity: 0.840
n_estimators=   500  RF R^2 effective diffusivity: 0.844  RF R^2 effective conductivity: 0.840


In [19]:
rf_final = RandomForestRegressor(n_estimators=200, min_samples_leaf=50, max_depth=8, n_jobs=-1, random_state=0)
rf_final.fit(X_train, y_train)
p = rf_final.predict(X_test)
print(f"RF R^2 FINAL effective diffusivity: {r2_score(y_test[:, 0], p[:, 0]):.3f}\n"
      f"RF R^2 FINAL effective conductivity: {r2_score(y_test[:, 1], p[:, 1]):.3f}")

RF R^2 FINAL effective diffusivity: 0.869
RF R^2 FINAL effective conductivity: 0.863
